In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd

from src.preprocessing.loader import DataLoader
from src.preprocessing.cleaner import DataCleaner
from src.preprocessing.encoder import DataEncoder
from src.preprocessing.scaler import DataScaler
from src.preprocessing.splitter import DataSplitter

from src.config.local_outlier_factor_config import LocalOutlierFactorConfig
from src.models.local_outlier_factor import LocalOutlierFactorModel

In [3]:
DATASET = "../datasets/processed/cicids2017.csv"

In [4]:
from src.preprocessing.loader import DataLoader

loader = DataLoader()

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", 
]

df = loader.load_multiple(files)

print(df.shape)

2026-07-28 15:24:04 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv started.
2026-07-28 15:24:04 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Monday-WorkingHours.pcap_ISCX.csv
2026-07-28 15:24:05 | INFO     | AdaptiveRL | Loaded Monday-WorkingHours.pcap_ISCX.csv | Shape=(529918, 79)
2026-07-28 15:24:05 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv completed in 1.7582 seconds.
2026-07-28 15:24:05 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv started.
2026-07-28 15:24:05 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Tuesday-WorkingHours.pcap_ISCX.csv
2026-07-28 15:24:07 | INFO     | AdaptiveRL | Loaded Tuesday-WorkingHours.pcap_ISCX.csv | Shape=(445909, 79)
2026-07-28 15:24:07 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv completed in 1.3242 seconds.
2026-07-28 15:24:07 | INFO     | AdaptiveRL | Lo

(2830743, 79)


In [5]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()

df = cleaner.clean(df)

print(df.shape)

2026-07-28 15:24:14 | INFO     | AdaptiveRL | Replacing Infinite Values started.
2026-07-28 15:24:15 | INFO     | AdaptiveRL | Replacing Infinite Values completed in 1.4670 seconds.
2026-07-28 15:24:15 | INFO     | AdaptiveRL | Removing Duplicates started.
2026-07-28 15:24:23 | INFO     | AdaptiveRL | Removed 308381 duplicate rows.
2026-07-28 15:24:23 | INFO     | AdaptiveRL | Removing Duplicates completed in 7.6189 seconds.
2026-07-28 15:24:23 | INFO     | AdaptiveRL | Removing Missing Values started.
2026-07-28 15:24:24 | INFO     | AdaptiveRL | Removed 1564 rows containing missing values.
2026-07-28 15:24:24 | INFO     | AdaptiveRL | Removing Missing Values completed in 0.6241 seconds.
2026-07-28 15:24:24 | INFO     | AdaptiveRL | Removing Constant Columns started.
2026-07-28 15:24:25 | INFO     | AdaptiveRL | Removed 8 constant columns.
2026-07-28 15:24:25 | INFO     | AdaptiveRL | Removing Constant Columns completed in 1.3028 seconds.
2026-07-28 15:24:25 | INFO     | AdaptiveRL | 

(2520798, 71)


In [6]:
df.columns = df.columns.str.strip()

In [7]:
from src.preprocessing.encoder import DataEncoder

encoder = DataEncoder(target_column=" Label")

df = encoder.fit_transform(df)

print(df.dtypes["Label"])
print(df["Label"].unique()[:10])

2026-07-28 15:24:29 | INFO     | AdaptiveRL | Encoding Dataset started.
2026-07-28 15:24:29 | INFO     | AdaptiveRL | Encoding Dataset completed in 0.2110 seconds.
2026-07-28 15:24:29 | INFO     | AdaptiveRL | Encoding completed.


int64
[ 0  7 11  6  5  4  3  8 12 14]


In [8]:
from src.preprocessing.scaler import DataScaler

scaler = DataScaler(
    method="standard",
    target_column="Label",
)

df = scaler.fit_transform(df)

print(df.shape)
print(df["Label"].dtype)
print(df["Label"].unique()[:10])

2026-07-28 15:24:33 | INFO     | AdaptiveRL | Scaler (standard) fitted on 70 feature columns.
2026-07-28 15:24:34 | INFO     | AdaptiveRL | Scaling Dataset started.
2026-07-28 15:24:33 | INFO     | AdaptiveRL | Scaling Dataset completed in 1.4222 seconds.
2026-07-28 15:24:33 | INFO     | AdaptiveRL | Scaling completed.


(2520798, 71)
int64
[ 0  7 11  6  5  4  3  8 12 14]


In [9]:
X = df.drop(columns=["Label"])
y = df["Label"]

print(f"Features shape : {X.shape}")
print(f"Labels shape   : {y.shape}")
print("\nLabel Distribution:")
print(y.value_counts().sort_index())

Features shape : (2520798, 70)
Labels shape   : (2520798,)

Label Distribution:
Label
0     2095057
1        1948
2      128014
3       10286
4      172846
5        5228
6        5385
7        5931
8          11
9          36
10      90694
11       3219
12       1470
13         21
14        652
Name: count, dtype: int64


In [10]:
config = LocalOutlierFactorConfig(
    n_neighbors=20,
    contamination=0.05,
    novelty=True,
)

In [11]:
model = LocalOutlierFactorModel(config)

model

2026-07-28 15:24:43 | INFO     | src.models.local_outlier_factor | Local Outlier Factor initialized.


LocalOutlierFactorModel(n_neighbors=20, contamination=0.05, novelty=True)

In [12]:
model.fit(X)

2026-07-28 15:24:45 | WARNING  | src.models.local_outlier_factor | Sampling 100000 rows for Local Outlier Factor training.
2026-07-28 15:24:46 | INFO     | src.models.local_outlier_factor | Training Local Outlier Factor...
2026-07-28 15:24:51 | INFO     | src.models.local_outlier_factor | Training completed.


LocalOutlierFactorModel(n_neighbors=20, contamination=0.05, novelty=True)

In [13]:
X_test = X[:5000]

predictions = model.predict(X_test)

predictions[:10]

array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])

In [14]:
scores = model.anomaly_score(X_test)

scores[:10]

array([ 1.71032348, 17.18779657, 44.35366088, 17.18655698, -2.15508799,
       -0.56577932, -2.20331122, -2.18960776, -1.94766594, -2.20331122])

In [15]:
unique, counts = np.unique(predictions, return_counts=True)

for label, count in zip(unique, counts):
    print(f"{label}: {count}")

0: 4715
1: 285


In [16]:
from pathlib import Path

save_dir = Path("trained_models")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "local_outlier_factor.joblib"

model.save(model_path)

print(f"Model saved to: {model_path}")

2026-07-28 15:25:07 | INFO     | src.models.local_outlier_factor | Local Outlier Factor saved -> trained_models/local_outlier_factor.joblib


Model saved to: trained_models/local_outlier_factor.joblib


In [19]:
from src.models.local_outlier_factor import LocalOutlierFactorModel
loaded_model = LocalOutlierFactorModel.load(model_path)

print(loaded_model)

2026-07-28 15:25:58 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- trained_models/local_outlier_factor.joblib


LocalOutlierFactorModel(n_neighbors=20, contamination=0.05, novelty=True)


In [20]:
loaded_predictions = loaded_model.predict(X_test)

np.array_equal(
    predictions,
    loaded_predictions,
)

True

In [21]:
loaded_scores = loaded_model.anomaly_score(X_test)

np.allclose(
    scores,
    loaded_scores,
)

True

In [27]:
from src.models.local_outlier_factor import LocalOutlierFactorModel


loaded_model = LocalOutlierFactorModel.load(model_path)



2026-07-28 15:28:08 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- trained_models/local_outlier_factor.joblib
